# Collaborative Filtering Recommender

## Project
An Explainable Knowledge Graph-Driven Hybrid Recommender for Intelligent Task Allocation in Distributed Onboarding Operations

## Purpose

This notebook develops the collaborative filtering component of the intelligent task allocation recommender.

Unlike the content-based baseline, which compares task requirements with agent profile attributes, collaborative filtering learns from historical task allocation and performance patterns.

The collaborative filtering approach will identify agents with similar historical performance patterns across onboarding task types and use these relationships to estimate how suitable an agent may be for a new task.

The current experiment uses synthetic development data while awaiting the approved industry dataset. The results are therefore treated as development evidence rather than final real-world performance.

## Experiment Information

- Run ID: CF_RUN_001
- Model: User-Based Collaborative Filtering
- Dataset Version: Synthetic Development Dataset v1
- Random Seed: 42
- Training Split: 70%
- Validation Split: 15%
- Test Split: 15%
- Historical Signal: Performance Rating
- Similarity Measure: Cosine Similarity
- Test data will remain separate during model development

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

## 1. Set Experiment Configuration

In [2]:
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

project_root = Path.cwd().parent

raw_data_path = project_root / "data" / "raw"
processed_data_path = project_root / "data" / "processed"

print("Project root:", project_root)
print("Raw data:", raw_data_path)
print("Processed data:", processed_data_path)

Project root: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation
Raw data: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\data\raw
Processed data: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\data\processed


## 2. Load Training and Validation Data

The collaborative filtering model is developed using only historical interactions from the training dataset.

The validation dataset will be used to evaluate the model during development.

The test dataset is loaded only to confirm that it exists and will remain unused until final evaluation.

In [3]:
train_df = pd.read_csv(
    processed_data_path / "train_interactions.csv"
)

validation_df = pd.read_csv(
    processed_data_path / "validation_interactions.csv"
)

test_df = pd.read_csv(
    processed_data_path / "test_interactions.csv"
)

agents_df = pd.read_csv(
    raw_data_path / "agents.csv"
)

tasks_df = pd.read_csv(
    raw_data_path / "tasks.csv"
)

print("Training interactions:", len(train_df))
print("Validation interactions:", len(validation_df))
print("Test interactions:", len(test_df))
print("Agents:", len(agents_df))
print("Tasks:", len(tasks_df))

Training interactions: 912
Validation interactions: 196
Test interactions: 196
Agents: 100
Tasks: 2000


In [4]:
train_df[
    [
        "assignment_id",
        "task_id",
        "agent_id",
        "task_type",
        "performance_rating"
    ]
].head(10)

,assignment_id,task_id,agent_id,task_type,performance_rating
0,AS1191,T1471,A067,M-PESA Onboarding,4.35
1,AS1494,T1860,A021,M-PESA Onboarding,4.19
2,AS1089,T1351,A033,M-PESA Onboarding,3.25
3,AS0314,T402,A079,KYC Verification,4.84
4,AS0711,T887,A048,Account Activation,3.97
5,AS0398,T504,A044,SIM Registration,4.85
6,AS0200,T269,A045,KYC Verification,3.36
7,AS1133,T1402,A088,M-PESA Onboarding,4.65
8,AS0592,T749,A046,KYC Verification,4.44
9,AS0092,T113,A036,KYC Verification,3.67


In [5]:
print("Missing performance ratings:")
print(train_df["performance_rating"].isnull().sum())

print("\nPerformance rating range:")
print(
    train_df["performance_rating"].min(),
    "to",
    train_df["performance_rating"].max()
)

print("\nTraining interactions by task type:")
print(train_df["task_type"].value_counts())

Missing performance ratings:
0

Performance rating range:
2.76 to 5.0

Training interactions by task type:
task_type
SIM Registration      232
Account Activation    228
M-PESA Onboarding     227
KYC Verification      225
Name: count, dtype: int64


### Interpretation

The training interaction dataset contains historical agent-task outcomes with performance ratings.

These interactions provide the behavioural evidence required for collaborative filtering.

Rather than treating each individual task as an independent item, historical interactions will be grouped by task type. This reduces sparsity and allows the model to identify similarities between agents based on their previous performance across different onboarding activities.

## 3. Build the Agent-Task Type Interaction Matrix

Collaborative filtering requires a structured representation of historical interactions.

For this project, each row represents an onboarding agent and each column represents an onboarding task type.

The value in each cell represents the agent's average historical performance rating for that task type.

A value of 0 indicates that no training interaction was observed for that agent and task type. It does not mean that the agent performed poorly.

In [6]:
interaction_matrix = (
    train_df
    .pivot_table(
        index="agent_id",
        columns="task_type",
        values="performance_rating",
        aggfunc="mean"
    )
)

# Include all agents, including those with no training interactions
interaction_matrix = interaction_matrix.reindex(
    agents_df["agent_id"]
)

interaction_matrix = interaction_matrix.fillna(0)

interaction_matrix.head(10)

task_type,Account Activation,KYC Verification,M-PESA Onboarding,SIM Registration
agent_id,,,,
A001,0.000000,3.590000,0.0,0.000000
A002,3.230000,3.372500,0.0,0.000000
A003,4.530000,4.504000,0.0,4.624000
A004,4.782222,0.000000,0.0,0.000000
A005,4.890000,0.000000,0.0,0.000000
A006,0.000000,4.855000,0.0,4.953750
A007,0.000000,5.000000,5.0,0.000000
A008,0.000000,4.313333,0.0,4.600000
A009,3.685000,3.756000,0.0,3.623333


### 3.1 Inspect the Interaction Matrix

The dimensions and coverage of the interaction matrix are checked before calculating agent similarity.

In [7]:
print(
    "Matrix shape:",
    interaction_matrix.shape
)

print(
    "Number of agents:",
    interaction_matrix.shape[0]
)

print(
    "Number of task types:",
    interaction_matrix.shape[1]
)

print(
    "\nTask types:"
)

print(
    interaction_matrix.columns.tolist()
)

Matrix shape: (100, 4)
Number of agents: 100
Number of task types: 4

Task types:
['Account Activation', 'KYC Verification', 'M-PESA Onboarding', 'SIM Registration']


In [8]:
observed_task_types_per_agent = (
    interaction_matrix
    .gt(0)
    .sum(axis=1)
)

observed_task_types_per_agent.describe()

count    100.000000
mean       1.830000
std        0.652191
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        4.000000
dtype: float64

In [9]:
print(
    "Agents with no training history:",
    (observed_task_types_per_agent == 0).sum()
)

print(
    "Agents with history in only one task type:",
    (observed_task_types_per_agent == 1).sum()
)

print(
    "Agents with history in multiple task types:",
    (observed_task_types_per_agent > 1).sum()
)

Agents with no training history: 0
Agents with history in only one task type: 30
Agents with history in multiple task types: 70


## 4. Calculate Agent Similarity

User-based collaborative filtering identifies agents with similar historical performance patterns.

Cosine similarity is used to compare the interaction vectors of agents.

A similarity score closer to 1 indicates that two agents have more similar historical task-performance patterns.

In [10]:
agent_similarity = cosine_similarity(
    interaction_matrix
)

agent_similarity_df = pd.DataFrame(
    agent_similarity,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

agent_similarity_df.head()

agent_id,A001,A002,A003,A004,A005,A006,A007,A008,A009,A010,...,A091,A092,A093,A094,A095,A096,A097,A098,A099,A100
agent_id,,,,,,,,,,,,,,,,,,,,,
A001,1.000000,0.722200,0.571142,0.000000,0.000000,0.699952,0.707107,0.684011,0.587914,1.000000,...,0.000000,0.000000,0.000000,0.783485,0.568293,1.000000,0.000000,0.000000,0.000000,0.000000
A002,0.722200,1.000000,0.809809,0.691684,0.691684,0.505506,0.510672,0.493993,0.823556,0.722200,...,0.480945,0.691684,0.691684,0.995653,0.811543,0.722200,0.000000,0.691684,0.000000,0.000000
A003,0.571142,0.809809,1.000000,0.574439,0.574439,0.818544,0.403858,0.818400,0.999672,0.571142,...,0.820835,0.574439,0.574439,0.804444,0.657705,0.571142,0.586359,0.574439,0.420904,0.586359
A004,0.000000,0.691684,0.574439,1.000000,1.000000,0.000000,0.000000,0.000000,0.576801,0.000000,...,0.695324,1.000000,1.000000,0.621410,0.579921,0.000000,0.000000,1.000000,0.000000,0.000000
A005,0.000000,0.691684,0.574439,1.000000,1.000000,0.000000,0.000000,0.000000,0.576801,0.000000,...,0.695324,1.000000,1.000000,0.621410,0.579921,0.000000,0.000000,1.000000,0.000000,0.000000


In [11]:
example_agent = interaction_matrix.index[0]

similar_agents = (
    agent_similarity_df[
        example_agent
    ]
    .drop(
        example_agent
    )
    .sort_values(
        ascending=False
    )
    .head(5)
)

print(
    "Example Agent:",
    example_agent
)

print(
    "\nMost similar agents:"
)

similar_agents

Example Agent: A001

Most similar agents:


agent_id
A010    1.0
A024    1.0
A039    1.0
A045    1.0
A052    1.0
Name: A001, dtype: float64

### Interpretation

The agent-task type interaction matrix summarizes historical performance patterns from the training dataset.

Cosine similarity was then used to identify agents whose previous task performance patterns are similar.

These similarity relationships will allow the collaborative filtering model to estimate an agent's expected suitability for a task type based on the historical performance of similar agents.

Agents with limited or no historical interactions represent a cold-start limitation of collaborative filtering. This limitation motivates combining collaborative filtering with content-based and knowledge graph information in the final hybrid recommender.

## 5. Predict Agent Performance Using Collaborative Filtering

For a selected onboarding task type, the collaborative filtering model estimates how well each agent may perform.

The prediction is based on the historical performance ratings of similar agents who previously handled the same type of task.

Agents with higher similarity contribute more strongly to the predicted score.

Where insufficient historical information is available, the average training performance for the task type is used as a fallback.

In [12]:
# Keep the original matrix with missing values
interaction_matrix_raw = (
    train_df
    .pivot_table(
        index="agent_id",
        columns="task_type",
        values="performance_rating",
        aggfunc="mean"
    )
    .reindex(agents_df["agent_id"])
)

task_type_mean_ratings = (
    train_df
    .groupby("task_type")["performance_rating"]
    .mean()
)

task_type_mean_ratings

task_type
Account Activation    4.287719
KYC Verification      4.217067
M-PESA Onboarding     4.339339
SIM Registration      4.321810
Name: performance_rating, dtype: float64

In [13]:
def predict_cf_rating(
    agent_id,
    task_type,
    n_neighbors=10
):

    similarities = (
        agent_similarity_df[agent_id]
        .drop(agent_id)
        .sort_values(ascending=False)
    )

    weighted_sum = 0.0
    similarity_sum = 0.0

    neighbors_used = 0

    for neighbor_id, similarity in similarities.items():

        if neighbors_used >= n_neighbors:
            break

        if (
            neighbor_id in interaction_matrix_raw.index
            and task_type in interaction_matrix_raw.columns
        ):

            neighbor_rating = (
                interaction_matrix_raw.loc[
                    neighbor_id,
                    task_type
                ]
            )

            if pd.notna(neighbor_rating) and similarity > 0:

                weighted_sum += (
                    similarity * neighbor_rating
                )

                similarity_sum += similarity

                neighbors_used += 1

    if similarity_sum > 0:

        predicted_rating = (
            weighted_sum / similarity_sum
        )

    else:

        predicted_rating = (
            task_type_mean_ratings.get(
                task_type,
                train_df["performance_rating"].mean()
            )
        )

    return round(
        float(predicted_rating),
        4
    )

In [14]:
example_task_type = (
    train_df.iloc[0]["task_type"]
)

example_cf_agent = (
    agents_df.iloc[0]["agent_id"]
)

print("Agent:", example_cf_agent)
print("Task Type:", example_task_type)

print(
    "Predicted Rating:",
    predict_cf_rating(
        example_cf_agent,
        example_task_type
    )
)

Agent: A001
Task Type: M-PESA Onboarding
Predicted Rating: 4.1981


## 6. Generate Collaborative Filtering Recommendations

The predicted performance ratings are used to rank agents for a selected onboarding task.

For operational recommendations, unavailable agents can be excluded.

During historical validation, current availability is not enforced because current availability does not represent the agent's availability at the time of the historical assignment.

In [15]:
agents_lookup = (
    agents_df
    .set_index("agent_id")
    .to_dict("index")
)

tasks_lookup = (
    tasks_df
    .set_index("task_id")
    .to_dict("index")
)

In [16]:
def recommend_agents_cf(
    task_id,
    top_k=5,
    n_neighbors=10,
    enforce_availability=True
):

    task = tasks_lookup[task_id]

    task_type = task["task_type"]

    recommendations = []

    for agent_id in agents_df["agent_id"]:

        agent = agents_lookup[agent_id]

        if (
            enforce_availability
            and not bool(agent["availability"])
        ):
            continue

        predicted_rating = predict_cf_rating(
            agent_id,
            task_type,
            n_neighbors=n_neighbors
        )

        recommendations.append({
            "agent_id": agent_id,
            "predicted_rating": predicted_rating
        })

    recommendations_df = pd.DataFrame(
        recommendations
    )

    recommendations_df = (
        recommendations_df
        .sort_values(
            "predicted_rating",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    recommendations_df.insert(
        0,
        "rank",
        range(
            1,
            len(recommendations_df) + 1
        )
    )

    return recommendations_df

In [17]:
pending_tasks = tasks_df[
    tasks_df["status"] == "Pending"
]

sample_cf_task_id = (
    pending_tasks.iloc[0]["task_id"]
)

print(
    "Task:",
    sample_cf_task_id
)

print(
    "Task Type:",
    tasks_lookup[
        sample_cf_task_id
    ]["task_type"]
)

recommend_agents_cf(
    sample_cf_task_id,
    top_k=5
)

Task: T004
Task Type: KYC Verification


,rank,agent_id,predicted_rating
0,1,A019,4.5950
1,2,A080,4.5936
2,3,A090,4.5758
3,4,A020,4.5439
4,5,A008,4.5358


## 7. Evaluate Collaborative Filtering

The collaborative filtering model is evaluated using the same validation dataset used for the content-based baseline.

For each validation task, the historically assigned agent is treated as the relevant agent.

The test dataset remains unused.

In [18]:
def evaluate_cf_task(
    task_id,
    relevant_agent_id,
    k=5
):

    recommendations = recommend_agents_cf(
        task_id,
        top_k=k,
        enforce_availability=False
    )

    recommended_agents = (
        recommendations[
            "agent_id"
        ].tolist()
    )

    if relevant_agent_id in recommended_agents:

        rank = (
            recommended_agents.index(
                relevant_agent_id
            ) + 1
        )

        hit = 1
        recall = 1.0
        precision = 1.0 / k
        average_precision = 1.0 / rank

    else:

        rank = None
        hit = 0
        recall = 0.0
        precision = 0.0
        average_precision = 0.0

    return {
        "task_id": task_id,
        "relevant_agent_id": relevant_agent_id,
        "hit": hit,
        "precision": precision,
        "recall": recall,
        "average_precision": average_precision,
        "relevant_agent_rank": rank
    }

In [19]:
def evaluate_cf_dataset(
    validation_data,
    k=5
):

    results = []

    for _, row in validation_data.iterrows():

        result = evaluate_cf_task(
            row["task_id"],
            row["agent_id"],
            k=k
        )

        results.append(result)

    return pd.DataFrame(results)

In [20]:
cf_evaluation_summary = []

for k in [1, 3, 5]:

    cf_results = evaluate_cf_dataset(
        validation_df,
        k=k
    )

    cf_evaluation_summary.append({
        "K": k,

        "Precision@K":
            cf_results[
                "precision"
            ].mean(),

        "Recall@K":
            cf_results[
                "recall"
            ].mean(),

        "Hit Rate@K":
            cf_results[
                "hit"
            ].mean(),

        "MAP@K":
            cf_results[
                "average_precision"
            ].mean()
    })


cf_evaluation_summary_df = pd.DataFrame(
    cf_evaluation_summary
)

cf_evaluation_summary_df

,K,Precision@K,Recall@K,Hit Rate@K,MAP@K
0,1,0.015306,0.015306,0.015306,0.015306
1,3,0.017007,0.051020,0.051020,0.030612
2,5,0.018367,0.091837,0.091837,0.040051


In [21]:
content_metrics = pd.read_csv(
    project_root
    / "experiments"
    / "CBF_RUN_001"
    / "validation_metrics.csv"
)

model_comparison_df = pd.DataFrame({
    "K": cf_evaluation_summary_df["K"],

    "Content-Based Hit Rate": (
        content_metrics[
            "Hit Rate@K"
        ]
    ),

    "Collaborative Filtering Hit Rate": (
        cf_evaluation_summary_df[
            "Hit Rate@K"
        ]
    ),

    "Content-Based MAP": (
        content_metrics[
            "MAP@K"
        ]
    ),

    "Collaborative Filtering MAP": (
        cf_evaluation_summary_df[
            "MAP@K"
        ]
    )
})

model_comparison_df

,K,Content-Based Hit Rate,Collaborative Filtering Hit Rate,Content-Based MAP,Collaborative Filtering MAP
0,1,0.091837,0.015306,0.091837,0.015306
1,3,0.209184,0.051020,0.138605,0.030612
2,5,0.255102,0.091837,0.149575,0.040051


### Interpretation

The collaborative filtering results are compared with the content-based baseline using the same validation interactions.

This comparison shows whether historical agent-task performance patterns provide additional recommendation value beyond profile-based matching.

Collaborative filtering also demonstrates an important limitation: agents with limited historical interaction data have less behavioural information available for recommendation.

This cold-start limitation is one reason the final system combines collaborative filtering with content-based and knowledge graph information.

In [25]:
# ---------------------------------------------------------
# Save Collaborative Filtering Experiment Evidence
# ---------------------------------------------------------

import json
from pathlib import Path


# Create experiment folder
cf_experiment_path = (
    project_root
    / "experiments"
    / "CF_RUN_001"
)

cf_experiment_path.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 1. Save experiment configuration
# ---------------------------------------------------------

cf_config = {
    "run_id": "CF_RUN_001",
    "model": "User-Based Collaborative Filtering",
    "dataset_version": "synthetic_v1",
    "random_seed": 42,
    "similarity_measure": "cosine similarity",
    "neighbors": 10,
    "interaction_signal": "performance_rating",
    "training_split": 0.70,
    "validation_split": 0.15,
    "test_split": 0.15,
    "evaluation_data": "validation",
    "test_data_used": False
}

config_file = (
    cf_experiment_path
    / "config.json"
)

with open(
    config_file,
    "w"
) as file:

    json.dump(
        cf_config,
        file,
        indent=4
    )


# ---------------------------------------------------------
# 2. Save validation metrics
# ---------------------------------------------------------

validation_metrics_file = (
    cf_experiment_path
    / "validation_metrics.csv"
)

cf_evaluation_summary_df.to_csv(
    validation_metrics_file,
    index=False
)


# ---------------------------------------------------------
# 3. Save comparison with Content-Based model
# ---------------------------------------------------------

comparison_file = (
    cf_experiment_path
    / "model_comparison.csv"
)

model_comparison_df.to_csv(
    comparison_file,
    index=False
)


# ---------------------------------------------------------
# 4. Save Top-5 validation results
# ---------------------------------------------------------

# The previous evaluation loop ended with K = 5,
# therefore cf_results already contains the Top-5 results.

cf_results_k5 = cf_results.copy()

predictions_file = (
    cf_experiment_path
    / "validation_predictions_k5.csv"
)

cf_results_k5.to_csv(
    predictions_file,
    index=False
)


# ---------------------------------------------------------
# 5. Confirm files were saved
# ---------------------------------------------------------

print(
    "Collaborative filtering experiment evidence saved successfully."
)

print(
    "\nExperiment folder:",
    cf_experiment_path
)

print("\nSaved files:")

for file in cf_experiment_path.iterdir():
    print("-", file.name)

Collaborative filtering experiment evidence saved successfully.

Experiment folder: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\experiments\CF_RUN_001

Saved files:
- config.json
- model_comparison.csv
- validation_metrics.csv
- validation_predictions_k5.csv


## Conclusion

The collaborative filtering component was successfully developed using historical agent-task performance interactions.

Cosine similarity was used to identify agents with similar historical performance patterns, and these relationships were used to estimate agent suitability for different onboarding task types.

The collaborative filtering model achieved a Hit Rate@5 of approximately 9.18% on the synthetic validation dataset.

The content-based baseline achieved a higher Hit Rate@5 of approximately 25.51%, indicating that agent profile information currently provides stronger recommendation evidence than historical interactions alone.

The collaborative filtering results still provide useful behavioural information that can later be combined with content-based and knowledge graph information in the hybrid recommender.

The test dataset remained unused during model development and is reserved for later final evaluation.